# Annualize emissions from baseline year to target year for each climactor_id

In [22]:
from pathlib import Path

import numpy as np
import pandas as pd
import os

INPUT_PATH = Path("data_raw/climactor_targets_combined.csv")
OUTPUT_PATH = Path("data_inter/city_points.csv")

merged_unique_check = pd.read_csv(INPUT_PATH)

for col in ["inv_year", "total_emissions", "inv_value"]:
    if col not in merged_unique_check.columns:
        merged_unique_check[col] = np.nan

merged_unique_check.head()

,baseline_year,baseline_value,target_type,target_year,target_value,target_unit,baseline_unit,climactor_id,datasource_id,GDAM_id,iso,entity_type,inv_year,total_emissions,inv_value
0,2011,43491.3,Absolute emission reduction,2030,40.0,percent,tCO2,00732a73f4f159b3,JRC:GCoM - MyCovenant:2023,BEL.2.2.1.5_1,BEL,City,NaN,NaN,NaN
1,2011,48759.2,Absolute emission reduction,2030,40.0,percent,tCO2,00887dae5af7b824,JRC:GCoM - MyCovenant:2023,BEL.2.1.3.22_1,BEL,City,NaN,NaN,NaN
2,2011,192.1,Absolute emission reduction,2030,45.0,percent,tCO2,009e873fc6c97fdc,JRC:GCoM - MyCovenant:2023,PRT.10.3_1,PRT,SubRegion,NaN,NaN,NaN
3,2011,192.1,Absolute emission reduction,2050,80.0,percent,tCO2,009e873fc6c97fdc,JRC:GCoM - MyCovenant:2023,PRT.10.3_1,PRT,SubRegion,NaN,NaN,NaN
4,2011,8965.4,Absolute emission reduction,2030,40.0,percent,tCO2,00b3f9e9c6778784,JRC:GCoM - MyCovenant:2023,ITA.19.84.007,ITA,City,NaN,NaN,NaN


## Capture all the emission records form baseline year to target year

In [ ]:
def first_non_na(series):
    """Return the first non-NA value in a series, or NaN if all are NA."""
    non_na = series.dropna()
    return non_na.iloc[0] if len(non_na) > 0 else np.nan

def compute_city_points(df, group_keys=["iso", "entity_type", "climactor_id"]):
    """
    Aggregate to one row per entity, compute target emissions, and flag anomalies.
    """
    agg = (
        df.groupby(group_keys, as_index=False)
        .agg(
            byear=("baseline_year",   first_non_na),
            bval =("baseline_value",  first_non_na),
            eyear=("inv_year",        first_non_na),
            eval =("total_emissions", first_non_na),
            # Include target information
            target_year  =("target_year",  first_non_na),
            target_value =("target_value", first_non_na),
        )
    )

    # Identify complete reduction targets (target_value == 100)
    agg["has_complete_reduction"] = agg["target_value"].notna() & (agg["target_value"] == 100)

    # Compute target emission based on target_value
    conditions = [
        # If target_value == 100, target emission is 0
        agg["has_complete_reduction"],
        # If have baseline and target_value != 0 and != 100, compute target emission
        agg["bval"].notna() & agg["target_value"].notna()
        & (agg["target_value"] > 0) & (agg["target_value"] < 100),
    ]
    choices = [
        0.0,
        agg["bval"] * (1 - agg["target_value"] / 100),
    ]
    agg["target_emission"] = np.select(conditions, choices, default=np.nan)

    # Flag: target year is before inventory year
    agg["T_before"] = np.where(
        agg["target_year"].notna() & agg["eyear"].notna() & (agg["target_year"] < agg["eyear"]),
        1, 0,
    )

    # Flag: target emission greater than total emissions
    agg["T_greater"] = np.where(
        agg["target_emission"].notna() & agg["eval"].notna() & (agg["target_emission"] > agg["eval"]),  # noqa: E501
        1, 0,
    )

    agg["target_record"] = agg["climactor_id"].astype(str) + "_" + agg["target_year"].astype(str)

    return agg


def build_next_points(df, prev_points, used_records):
    """
    For records not yet claimed, join against the previous round's target emissions
    to form a new baseline to target segment.
    """
    remaining = df[~df["target_record"].isin(used_records)].copy()

    joined = remaining.merge(
        prev_points[["climactor_id", "iso", "entity_type", "target_year", "target_emission", "target_value"]],
        on=["climactor_id", "iso", "entity_type"],
        suffixes=("_next", "_prev"),
    )

    # Previous round's target becomes the new baseline
    joined["baseline_year"]  = joined["target_year_prev"]
    joined["baseline_value"] = joined["target_emission"]
    joined["inv_year"]       = joined["target_year_prev"]
    joined["inv_value"]      = joined["target_emission"]
    joined["target_year"]    = joined["target_year_next"]
    joined["target_value"]   = joined["target_value_next"]

    cols = ["iso", "entity_type", "climactor_id",
            "baseline_value", "baseline_year", "inv_year",
            "total_emissions", "target_year", "target_value"]
    return compute_city_points(joined[cols])


# --- Round 1: anchor on original baseline + inventory data ---
city_points_1 = compute_city_points(merged_unique_check)

# --- Rounds 2–5: chain previous round's target emission as the next baseline ---
used = set(city_points_1["target_record"])

city_points_2 = build_next_points(merged_unique_check, city_points_1, used)
used |= set(city_points_2["target_record"])

city_points_3 = build_next_points(merged_unique_check, city_points_2, used)
used |= set(city_points_3["target_record"])

city_points_4 = build_next_points(merged_unique_check, city_points_3, used)
used |= set(city_points_4["target_record"])

city_points_5 = build_next_points(merged_unique_check, city_points_4, used)

# --- Stack all rounds ---
city_points = pd.concat(
    [city_points_1, city_points_2, city_points_3, city_points_4, city_points_5],
    ignore_index=True,
)

# Keep only rows with at least one year available
city_points = city_points[
    ~(city_points["byear"].isna() & city_points["eyear"].isna())
]  

print(city_points)# 3912 obs

      iso entity_type      climactor_id  byear          bval   eyear  eval  \
0     ALB      Region  cfbf67e6928d3b2b   2011  5.430545e+05     NaN   NaN   
1     ARE      Region  207c479f7332744a   2021  8.110044e+07     NaN   NaN   
2     ARG        City  0b8d0bfae5cd87bf   2018  7.570194e+05     NaN   NaN   
3     ARG        City  216c4c9667b5d334   2020  8.783634e+04     NaN   NaN   
4     ARG        City  240a4b68a269fd54   2017  1.236869e+05     NaN   NaN   
...   ...         ...               ...    ...           ...     ...   ...   
3907  CAN   SubRegion  38aa853d85b9b187   2040  5.039347e+04  2040.0   NaN   
3908  GBR        City  5ecab44f1122d755   2035  1.448842e+05  2035.0   NaN   
3909  GBR        City  e3525d45e4e3c73c   2035  2.965018e+04  2035.0   NaN   
3910  GBR        City  e966fe216227f564   2035  7.561940e+03  2035.0   NaN   
3911  USA        City  3ab3c919e8671ef4   2040  2.152867e+02  2040.0   NaN   

      target_year  target_value  has_complete_reduction  target

## Use linear function to predict annual emission

In [24]:
def predict_emissions(city_points: pd.DataFrame) -> pd.DataFrame | None:
    # climactor_id
    climactor_id = city_points["climactor_id"].iloc[0]

    # Sort by target_year and filter valid rows
    city_points = (
        city_points
        .sort_values("target_year")
        .loc[city_points["target_year"].notna() & city_points["target_emission"].notna()]
        .reset_index(drop=True)
    )

    if len(city_points) == 0:
        return None

    # Create prediction years sequence
    years = list(range(1990, 2051))
    predictions = pd.DataFrame({"year": years, "predicted_emission": np.nan})

    # Determine starting point: prefer eyear/eval, fallback to byear/bval
    if pd.notna(city_points["eyear"].iloc[0]) and pd.notna(city_points["eval"].iloc[0]):
        start_year     = city_points["eyear"].iloc[0]
        start_emission = city_points["eval"].iloc[0]
    elif pd.notna(city_points["byear"].iloc[0]) and pd.notna(city_points["bval"].iloc[0]):
        start_year     = city_points["byear"].iloc[0]
        start_emission = city_points["bval"].iloc[0]
    else:
        return None

    def _has_both_eye_and_bval(row0):
        """Both eyear/eval and byear/bval are present for this entity."""
        return (
            pd.notna(row0["eyear"]) and pd.notna(row0["eval"])
            and pd.notna(row0["byear"]) and pd.notna(row0["bval"])
        )

    def _fill_linear(predictions, slope, seg_start_year, seg_start_emission, target_year, target_emission):
        """
        Vectorised fill: flat before seg_start_year, linear in between,
        flat at target_emission from target_year onward.
        """
        yr = predictions["year"].to_numpy(dtype=float)
        em = np.where(
            yr <= seg_start_year,
            seg_start_emission,
            np.where(
                yr >= target_year,
                # Keep emission at target level after reaching target year
                target_emission,
                np.maximum(0, seg_start_emission + slope * (yr - seg_start_year)),
            ),
        )
        predictions["predicted_emission"] = em
        return predictions

    row0 = city_points.iloc[0]

    # --- Single-entry case ---
    if len(city_points) == 1:
        target_year      = row0["target_year"]
        target_emission  = row0["target_emission"]

        # Check if we should use bval -> target_emission line instead
        use_bval_line = False

        if _has_both_eye_and_bval(row0):
            # Condition 1: if target_year < eyear (report year)
            if target_year < row0["eyear"]:
                use_bval_line = True

            # Condition 2: if target_emission > eval
            if target_emission > row0["eval"]:
                use_bval_line = True

        if use_bval_line:
            # Use straight line from bval to target_emission
            bval_start_year      = row0["byear"]
            bval_start_emission  = row0["bval"]
            slope = (target_emission - bval_start_emission) / (target_year - bval_start_year)
            predictions = _fill_linear(predictions, slope, bval_start_year, bval_start_emission,
                                       target_year, target_emission)
        else:
            # Original single-entry logic
            slope = (target_emission - start_emission) / (target_year - start_year)
            predictions = _fill_linear(predictions, slope, start_year, start_emission,
                                       target_year, target_emission)

    # --- Multi-entry case ---
    else:
        # Check if we should use bval -> target_emission line instead
        use_bval_line = False

        if _has_both_eye_and_bval(row0):
            # Condition 1: if any target_year < eyear (report year)
            if (city_points["target_year"] < row0["eyear"]).any():
                use_bval_line = True

            # Condition 2: if any target_emission > eval
            if (city_points["target_emission"] > row0["eval"]).any():
                use_bval_line = True

        if use_bval_line:
            # Use straight line from bval to first target_emission
            target_year         = row0["target_year"]
            target_emission     = row0["target_emission"]
            bval_start_year     = row0["byear"]
            bval_start_emission = row0["bval"]
            slope = (target_emission - bval_start_emission) / (target_year - bval_start_year)
            predictions = _fill_linear(predictions, slope, bval_start_year, bval_start_emission,
                                       target_year, target_emission)

        else:
            # Original multi-entry logic
            points = (
                pd.DataFrame({
                    "year":     [start_year] + city_points["target_year"].tolist(),
                    "emission": [start_emission] + city_points["target_emission"].tolist(),
                })
                .drop_duplicates()
                .sort_values("year")
                .reset_index(drop=True)
            )

            yr_arr = predictions["year"].to_numpy(dtype=float)
            em_arr = np.empty(len(yr_arr))

            for i, year in enumerate(yr_arr):
                if year <= points["year"].iloc[0]:
                    em_arr[i] = points["emission"].iloc[0]
                elif year > points["year"].iloc[-1]:
                    # For years after the last target year, keep emission at last target level
                    em_arr[i] = points["emission"].iloc[-1]
                else:
                    segment_idx = (points["year"] <= year).sum() - 1
                    if points["emission"].iloc[segment_idx] == 0:
                        em_arr[i] = 0
                    elif segment_idx < len(points) - 1:
                        x1 = points["year"].iloc[segment_idx]
                        x2 = points["year"].iloc[segment_idx + 1]
                        y1 = points["emission"].iloc[segment_idx]
                        y2 = points["emission"].iloc[segment_idx + 1]
                        slope = (y2 - y1) / (x2 - x1)
                        em_arr[i] = max(0, y1 + slope * (year - x1))
                    else:
                        em_arr[i] = points["emission"].iloc[segment_idx]

            predictions["predicted_emission"] = em_arr

    # Add climactor_id and data_source info to predictions
    predictions["climactor_id"] = climactor_id
    predictions["data_source"] = (
        "emission_year"
        if pd.notna(row0["eyear"]) and pd.notna(row0["eval"])
        else "baseline_year"
    )

    return predictions

## Check all predictions

In [25]:
all_predictions = (
    city_points
    .groupby("climactor_id", group_keys=False)
    .apply(predict_emissions)
    .reset_index(drop=True)
)

print(all_predictions)

check = (
    all_predictions
    .merge(
        merged_unique_check[["climactor_id", "iso", "entity_type"]].drop_duplicates(),
        on="climactor_id",
        how="left",
    )
    .drop_duplicates()
    .groupby("entity_type", as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

all_predictions_wide = (
    all_predictions
    .merge(
        merged_unique_check[["climactor_id", "iso", "entity_type"]].drop_duplicates(),
        on="climactor_id",
        how="left",
    )
    .pivot_table(index=["climactor_id", "iso", "entity_type", "data_source"],
                 columns="year",
                 values="predicted_emission",
                 aggfunc="first")
    .reset_index()
    .rename_axis(None, axis=1)
)

/var/folders/_8/h00r367d04lc0df043_znnvc0000gn/T/ipykernel_99858/1484406694.py:84: RuntimeWarning: divide by zero encountered in scalar divide
  slope = (target_emission - start_emission) / (target_year - start_year)
/var/folders/_8/h00r367d04lc0df043_znnvc0000gn/T/ipykernel_99858/1484406694.py:50: RuntimeWarning: invalid value encountered in multiply
  np.maximum(0, seg_start_emission + slope * (yr - seg_start_year)),
/var/folders/_8/h00r367d04lc0df043_znnvc0000gn/T/ipykernel_99858/1484406694.py:84: RuntimeWarning: divide by zero encountered in scalar divide
  slope = (target_emission - start_emission) / (target_year - start_year)
/var/folders/_8/h00r367d04lc0df043_znnvc0000gn/T/ipykernel_99858/1484406694.py:50: RuntimeWarning: invalid value encountered in multiply
  np.maximum(0, seg_start_emission + slope * (yr - seg_start_year)),
/var/folders/_8/h00r367d04lc0df043_znnvc0000gn/T/ipykernel_99858/1484406694.py:84: RuntimeWarning: divide by zero encountered in scalar divide
  slope = (

        year  predicted_emission      climactor_id    data_source
0       1990          2629824.00  0033fb16f68dfeeb  baseline_year
1       1991          2629824.00  0033fb16f68dfeeb  baseline_year
2       1992          2629824.00  0033fb16f68dfeeb  baseline_year
3       1993          2629824.00  0033fb16f68dfeeb  baseline_year
4       1994          2629824.00  0033fb16f68dfeeb  baseline_year
...      ...                 ...               ...            ...
199953  2046             2093.64  ffe0efb6be3172e5  baseline_year
199954  2047             2093.64  ffe0efb6be3172e5  baseline_year
199955  2048             2093.64  ffe0efb6be3172e5  baseline_year
199956  2049             2093.64  ffe0efb6be3172e5  baseline_year
199957  2050             2093.64  ffe0efb6be3172e5  baseline_year

[199958 rows x 4 columns]


/var/folders/_8/h00r367d04lc0df043_znnvc0000gn/T/ipykernel_99858/344855117.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(predict_emissions)


In [26]:
os.makedirs("data_inter", exist_ok=True)
all_predictions_wide.to_csv("data_inter/all_predictions.csv", index=False)